# Pipeline de Extração de Features (Two-Stage)



## Contexto e objetivo

Este notebook faz parte de um trabalho cujo ponto de partida é o artigo
**"Decoding Reddit Memes Virality"** (Sah & Jordan, 2025), publicado no
*International Journal of Data Science and Analytics*. O artigo analisa
16.968 memes extraídos do Reddit com o objetivo de identificar preditores
de viralidade, ou seja, quais características visuais, textuais e temporais
tornam um meme mais propenso a se tornar viral.

Para isso, os autores construíram um pipeline de extração de features que combina:

- **Visão computacional** — detecção de objetos (YOLOv5), reconhecimento de
  expressões faciais (FER) e análise de cor (médias HSV/RGB);
- **NLP** — extração de texto via OCR (EasyOCR + OpenCV) e análise de sentimento
  com DistilBERT;
- **Análise temporal** — segmentação dos horários de postagem em UTC.

As features extraídas alimentam modelos CatBoost, LightGBM e XGBoost, sendo o
CatBoost o melhor desempenho com AUC-ROC de 0.77 na combinação de features
visuais e textuais.


## Nossa Aplicação

Nosso trabalho consiste em reimplementar esse pipeline com a adição de técnicas de processamento
de dados em escala, introduzindo técnicas que o artigo original não utiliza.

O artigo processa as imagens de forma inteiramente sequencial em Python puro:
uma imagem de cada vez, sem paralelismo. Nossa proposta substitui essa abordagem
por um pipeline distribuído baseado em **Apache Spark**, onde cada imagem é
processada como uma tarefa independente distribuída entre workers, viabilizando
a execução sobre o dataset completo de forma paralela e escalável.

Além da distribuição via Spark, expandimos o pipeline original com **embeddings
textuais** gerados pelo modelo `all-MiniLM-L6-v2` (Sentence-Transformers),
uma representação semântica densa do texto extraído por OCR que o artigo não
utiliza.

## Sobre este Notebook

O pipeline foi de fato executado em scripts Python standalone,
via terminal sobre um cluster Spark em ambiente Docker (`docker compose`). Os
scripts estão em `src/` e são a entrega técnica principal do projeto.

Este Colab existe para demonstrar e documentar a execução
do pipeline, reproduzindo as chamadas aos mesmos módulos. **Ele não substitui
a execução real** mas serve como referência legível de tudo o que o pipeline faz e produz.


## Arquitetura

### Estágio A — Detecção + Emoções + Cores

| Passo | Modelo / Técnica | Saída |
|---|---|---|
| 1. Object Detection | YOLOv5 (ultralytics) | Bounding boxes + labels por imagem |
| 2. Emotion Recognition | FER (Facial Emotion Recognition) | Emoção dominante por detecção |
| 3. Color Analysis | Estatísticas RGB/HSV por crop | Média, desvio padrão (R,G,B,H,S,V) |
| 4. Crop Persistence | OpenCV `cv2.imwrite` | Crops salvos em `/workspace/data/output/crops/` |

### Estágio B — OCR + Embeddings

| Passo | Modelo / Técnica | Saída |
|---|---|---|
| 5. OCR (crop-level) | EasyOCR | Texto extraído de cada crop |
| 6. OCR (global) | EasyOCR | Texto extraído da imagem completa (fallback) |
| 7. Text Embeddings | Sentence-Transformers (`all-MiniLM-L6-v2`) | Vetor float32 (384-d) |

### Pós-Processamento

| Passo | Descrição | Saída |
|---|---|---|
| 8. Explode JSON | Expande o JSON aninhado (uma linha por detecção) | `features.parquet` tabela normalizada |

---

**Pré-requisitos:** Cluster Spark em execução (`spark://spark-master:7077`),
GPUs disponíveis, modelos pré-carregados (`scripts/preload_models.py`).

## 0 — Verificação do Ambiente

Antes de executar qualquer estágio do pipeline, verificamos se as três condições
essenciais para o funcionamento estão satisfeitas.

A primeira é a disponibilidade das dependências Python. A segunda é a presença de GPU. O YOLOv5 e o FER operam com aceleração CUDA; sem
uma GPU disponível, o tempo de processamento por imagem aumenta significativamente, tornando a execução sobre o dataset completo inviável na prática.

A terceira é a conexão com o cluster Spark. O pipeline distribui o processamento
das imagens entre workers via `spark://spark-master:7077`. Essa conexão só existe se o ambiente Docker foi iniciado previamente com `docker compose up -d`. Caso
contrário, a célula retornará erro e os estágios seguintes não poderão ser executados.

Por fim, mapeamos os caminhos de entrada e saída utilizados ao longo do pipeline
e fazemos uma contagem inicial das imagens disponíveis por subreddit, antes de iniciar o processamento.

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
from PIL import Image

sys.path.insert(0, '/workspace/src')

sns.set(style='whitegrid')
warnings.filterwarnings('ignore')
%matplotlib inline

print('Imports OK')

In [ ]:
import torch

print(f'CUDA disponivel: {torch.cuda.is_available()}')
print(f'GPUs detectadas: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  [{i}] {torch.cuda.get_device_name(i)}')

try:
    from pyspark.sql import SparkSession
    spark_test = SparkSession.builder \
        .appName('SparkConnectTest') \
        .master('spark://spark-master:7077') \
        .getOrCreate()
    print(f'\nSpark conectado: {spark_test.version}')
    spark_test.stop()
except Exception as e:
    print(f'\nERRO: Nao foi possivel conectar ao Spark: {e}')
    print('Certifique-se de que docker compose up -d foi executado.')

In [ ]:
# Caminhos (mesmos defaults dos scripts em src/)
IMAGES_DIR = '/workspace/data/images'
OUTPUT_PATH = '/workspace/data/output/features'
CROPS_ROOT = '/workspace/data/output/crops'
EASYOCR_MODEL_DIR = '/workspace/data/easyocr_models'
MODEL_NAME = 'yolov5n'

STAGE_A_PARQUET = f'{OUTPUT_PATH}/stage_a_raw_parquet'
STAGE_B_PARQUET = f'{OUTPUT_PATH}/features_raw_parquet'
FINAL_PARQUET = f'{OUTPUT_PATH}.parquet'

for name, p in [
    ('IMAGES_DIR', IMAGES_DIR),
    ('OUTPUT_PATH', OUTPUT_PATH),
    ('CROPS_ROOT', CROPS_ROOT),
    ('EASYOCR_MODEL_DIR', EASYOCR_MODEL_DIR),
]:
    exists = os.path.exists(p)
    print(f'{name}: {p}  →  exists={exists}')

In [ ]:
# Contagem rápida de imagens disponiveis
image_extensions = ('.jpg', '.jpeg', '.png')
all_images = []
for root, dirs, files in os.walk(IMAGES_DIR):
    for f in files:
        if f.lower().endswith(image_extensions):
            all_images.append(os.path.join(root, f))

print(f'Total de imagens encontradas: {len(all_images)}')

subreddits = set()
for p in all_images:
    rel = os.path.relpath(p, IMAGES_DIR)
    parts = rel.split(os.sep)
    if len(parts) >= 2:
        subreddits.add(parts[0])
print(f'Subreddits detectados: {len(subreddits)}')
if subreddits:
    print(f'  {sorted(subreddits)}')

---
## Estágio A — YOLO + FER + Cores + Crops

Este estágio carrega todas as imagens via Spark `binaryFile`, extrai caminho/subreddit/nome do arquivo,
e aplica um UDF (User-Defined Function). O uso de UDF é o que permite que esse
processamento ocorra de forma distribuída: o Spark divide o conjunto de imagens entre os workers disponíveis e cada worker executa a função de forma independente sobre sua partição.

**Toda a lógica de processamento está implementada nos scripts em `src/`. A célula abaixo apenas importa e chama esse módulo, passando os caminhos de entrada e saída configurados anteriormente. A implementação
em si está no script, não neste notebook.**

1. **YOLOv5** —
O primeiro passo dentro do UDF é a detecção de objetos com **YOLOv5n**. O modelo
recebe a imagem e retorna uma lista de bounding boxes, cada uma com seu label e score de confiança. A variante nano (`yolov5n`) foi escolhida pelo equilíbrio entre velocidade de inferência e acurácia suficiente para o tamanho e natureza das imagens de memes.
Esse passo replica diretamente o uso de YOLOv5 descrito na Seção 3.2 do artigo.
2. **FER** — Para cada objeto detectado, o recorte correspondente é extraído da imagem original e passado ao **FER (Facial Emotion Recognition)**. O FER tenta classificar a expressão facial presente no crop em uma das sete categorias usadas no artigo: raiva, desgosto, medo, felicidade, neutralidade, tristeza e surpresa. Nos casos em que o crop não contém um rosto detectável, o campo é marcado como erro, normal para animais e objetos.
3. **Color stats** — Em paralelo à análise de emoções, calculamos **estatísticas de cor** sobre cada bounding box: médias e desvios padrão dos canais RGB e HSV. Essas métricas reproduzem
as features `Average Hue`, `Average Saturation` e `Average Value` que o artigo aponta como preditores relevantes de viralidade, com `Average Saturation` apresentando coeficiente negativo e `Average Value` positivo: memes mais claros e menos saturados tendem a performar melhor.
4. **Crop persistence** — cada recorte é **salvo em disco** via OpenCV. Essa persistência é necessária porque o Estágio B, executado em seguida, precisa acessar os crops para aplicar OCR em nível de objeto, o que não seria possível se os recortes existissem apenas em memória durante a execução do Estágio A.

A saída é salva como Parquet particionado por subreddit.



Em paralelo à análise de emoções, calculamos **estatísticas de cor** sobre cada
bounding box: médias e desvios padrão dos canais RGB e HSV. Essas métricas reproduzem
as features `Average Hue`, `Average Saturation` e `Average Value` que o artigo aponta
como preditores relevantes de viralidade, com `Average Saturation` apresentando
coeficiente negativo e `Average Value` positivo — memes mais claros e menos saturados
tendem a performar melhor.

Por fim, cada recorte é **salvo em disco** via OpenCV. Essa persistência é necessária
porque o Estágio B, executado separadamente, precisa acessar os crops para aplicar OCR
em nível de objeto — algo que não seria possível se os recortes existissem apenas em
memória durante a execução do Estágio A.

A saída de todo esse processamento é gravada como **Parquet particionado por subreddit**,
o que permite que etapas subsequentes leiam apenas os subreddits de interesse sem
carregar o dataset inteiro na memória.

In [ ]:
from feature_pipeline_stage_a import run_pipeline as run_stage_a

# Garantir que diretorios de saida existam
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CROPS_ROOT, exist_ok=True)

print('Executando Estagio A...')
print(f'  images_dir = {IMAGES_DIR}')
print(f'  output_path = {OUTPUT_PATH}')
print(f'  model_name = {MODEL_NAME}')
print(f'  crops_root = {CROPS_ROOT}')

run_stage_a(
    images_dir=IMAGES_DIR,
    output_path=OUTPUT_PATH,
    model_name=MODEL_NAME,
    crops_root=CROPS_ROOT,
)
print('\nEstagio A concluido.')

### Diagnóstico — Resultados do Estágio A

In [ ]:
def safe_json_load(x):
    if x is None:
        return []
    if isinstance(x, (list, dict)):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            return json.loads(x)
        except Exception:
            return []
    return []


def explode_json_column(df, col_name):
    rows = []
    parsed = df[col_name].apply(safe_json_load)
    for i, arr in parsed.items():
        base = {
            'path': df.at[i, 'path'] if 'path' in df.columns else None,
            'filename': df.at[i, 'filename'] if 'filename' in df.columns else None,
            'subreddit': df.at[i, 'subreddit'] if 'subreddit' in df.columns else None,
        }
        for rec in arr if isinstance(arr, list) else []:
            if isinstance(rec, dict):
                rows.append({**base, **rec})
    return pd.DataFrame(rows)

In [ ]:
if os.path.exists(STAGE_A_PARQUET):
    df_a_raw = pd.read_parquet(STAGE_A_PARQUET)
    print(f'Stage A Parquet carregado: {len(df_a_raw)} linhas, colunas={df_a_raw.columns.tolist()}')

    if 'stage_a_json' in df_a_raw.columns:
        df_a = explode_json_column(df_a_raw, 'stage_a_json')
        print(f'Explodido para {len(df_a)} deteccoes individuais.')

        display(df_a.head(3))

        print(f'\n--- Status das deteccoes (Estagio A) ---')
        if 'status' in df_a.columns:
            print(df_a['status'].value_counts(dropna=False))

        print(f'\n--- Top-10 labels (YOLO) ---')
        if 'label' in df_a.columns:
            vc = df_a['label'].fillna('<<NA>>').value_counts().head(10)
            print(vc)

            plt.figure(figsize=(10, 4))
            sns.barplot(x=vc.index.astype(str), y=vc.values)
            plt.xticks(rotation=45, ha='right')
            plt.title('Top-10 labels detectadas (YOLO) — Estagio A')
            plt.tight_layout()
            plt.show()
else:
    print('Stage A Parquet nao encontrado. Execute a celula anterior primeiro.')

In [ ]:
# Amostra visual: imagens originais com bounding boxes
if os.path.exists(STAGE_A_PARQUET) and 'df_a' in locals() and len(df_a) > 0:
    sample = df_a[df_a['bbox'].notna()].sample(min(6, len(df_a)), random_state=42)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, sample.iterrows()):
        img_path = row.get('path') or row.get('image_path', '')
        local = img_path.replace('file://', '').replace('file:', '')
        try:
            img = Image.open(local).convert('RGB')
            ax.imshow(img)

            bbox = row.get('bbox')
            if bbox and isinstance(bbox, (list, np.ndarray)):
                x1, y1, x2, y2 = bbox
                rect = Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor='red', facecolor='none')
                ax.add_patch(rect)
                label = row.get('label', '?')
                conf = row.get('conf', '')
                ax.set_title(f'{label} conf={conf:.3f}' if isinstance(conf, float) else label,
                             fontsize=10, color='red')
            ax.axis('off')
        except Exception:
            ax.text(0.5, 0.5, f'Erro ao carregar:\n{local}',
                    ha='center', va='center', fontsize=8)
            ax.axis('off')

    plt.suptitle('Amostra de deteccoes — Bounding boxes (Estagio A)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Sem dados para visualizar.')

---
## Estágio B — OCR + Embeddings Textuais

O Estágio B enriquece cada detecção produzida pelo Estágio A com informação textual, lendo os
crops salvos em disco.

**OCR em dois níveis:**
- *Crop-level:* o EasyOCR processa cada recorte individualmente, capturando texto sobreposto
  diretamente sobre o objeto detectado.
- *Global (fallback):* caso o crop não contenha texto, o OCR é aplicado sobre a imagem completa,
  aumentando a cobertura total.

**Essa estratégia em dois níveis é uma extensão em relação ao artigo original, que aplicava OCR apenas globalmente via OpenCV + EasyOCR.**

**Embeddings textuais:**  
O texto extraído é vetorizado com o modelo `all-MiniLM-L6-v2` (Sentence-Transformers), gerando um vetor denso de 384 dimensões por detecção. Esses embeddings permitem análises de similaridade semântica e podem servir como features para modelos downstream, algo não presente no artigo original, que se limitava a análise de sentimento com DistilBERT.

A saída é gravada como Parquet, mantendo a estrutura particionada por subreddit.

In [ ]:
from feature_pipeline_stage_b import run_pipeline as run_stage_b

print('Executando Estagio B...')
print(f'  stage_a_input = {STAGE_A_PARQUET}')
print(f'  output_path = {OUTPUT_PATH}')
print(f'  model_dir = {EASYOCR_MODEL_DIR}')
print(f'  repartition = 16')

run_stage_b(
    stage_a_input=STAGE_A_PARQUET,
    output_path=OUTPUT_PATH,
    easyocr_model_dir=EASYOCR_MODEL_DIR,
    easyocr_download=False,
    repartition=16,
)
print('\nEstagio B concluido.')

### Diagnóstico — Resultados do Estágio B

In [ ]:
if os.path.exists(STAGE_B_PARQUET):
    df_b_raw = pd.read_parquet(STAGE_B_PARQUET)
    print(f'Stage B Parquet carregado: {len(df_b_raw)} imagens processadas')

    if 'features_json' in df_b_raw.columns:
        df_b = explode_json_column(df_b_raw, 'features_json')
        print(f'Explodido para {len(df_b)} deteccoes individuais.')

        # Estatisticas de OCR
        print(f'\n--- OCR: Cobertura de texto ---')
        if 'ocr_text' in df_b.columns:
            has_ocr = df_b['ocr_text'].fillna('').str.strip() != ''
            pct = float(has_ocr.mean()) * 100
            print(f'  ocr_text (crop-level) nao-vazio: {pct:.1f}% ({has_ocr.sum()}/{len(has_ocr)})')
        if 'ocr_global_text' in df_b.columns:
            has_ocr_g = df_b['ocr_global_text'].fillna('').str.strip() != ''
            pct_g = float(has_ocr_g.mean()) * 100
            print(f'  ocr_global_text (global) nao-vazio: {pct_g:.1f}% ({has_ocr_g.sum()}/{len(has_ocr_g)})')

        # Embeddings
        print(f'\n--- Embeddings ---')
        if 'embedding' in df_b.columns:
            def emb_dim(x):
                try:
                    return np.array(x, dtype=np.float32).size
                except Exception:
                    return None
            dims = df_b['embedding'].apply(emb_dim).dropna().unique()
            print(f'  Dimensao do embedding: {dims}')

            def emb_nonzero(x):
                try:
                    v = np.array(x, dtype=np.float32)
                    return v.size > 0 and np.any(v != 0)
                except Exception:
                    return False
            nz = df_b['embedding'].apply(emb_nonzero)
            print(f'  Embeddings nao-nulos: {float(nz.mean()):.1%} ({int(nz.sum())}/{len(nz)})')

        # Amostra de OCR
        print(f'\n--- Amostra de OCR text (10 exemplos) ---')
        if 'ocr_text' in df_b.columns:
            nonempty = df_b[df_b['ocr_text'].fillna('').str.strip() != '']
            if len(nonempty) > 0:
                cols = [c for c in ['filename', 'label', 'ocr_text'] if c in df_b.columns]
                display(nonempty[cols].head(10))
else:
    print('Stage B Parquet nao encontrado. Execute a celula anterior primeiro.')

In [ ]:
# Amostras visuais: crops com OCR detectado
if os.path.exists(STAGE_B_PARQUET) and 'df_b' in locals() and len(df_b) > 0 \
   and 'ocr_text' in df_b.columns and 'crop_path' in df_b.columns:
    has_text = df_b['ocr_text'].fillna('').str.strip() != ''
    has_crop = df_b['crop_path'].notna()
    candidates = df_b[has_text & has_crop]

    if len(candidates) > 0:
        sample = candidates.sample(min(6, len(candidates)), random_state=7)
        fig, axes = plt.subplots(2, 3, figsize=(12, 8))
        axes = axes.flatten()

        for ax, (_, row) in zip(axes, sample.iterrows()):
            cp = row.get('crop_path')
            if cp and os.path.isfile(cp):
                img = Image.open(cp).convert('RGB')
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, 'crop nao encontrado', ha='center', va='center')
            label = row.get('label') or ''
            ocr = row.get('ocr_text', '')[:50]
            ax.set_title(f'{label}\nOCR: {ocr}', fontsize=9)
            ax.axis('off')

        plt.suptitle('Amostra de crops com OCR detectado (Estagio B)', fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        print('Nenhum crop com texto OCR encontrado para exibir.')
else:
    print('Dados insuficientes para visualizacao de crops.')

---
## Pós-Processamento — Normalização

O Estágio B produz, para cada imagem, um JSON aninhado com a lista de todas as detecções. O pós-processador **explode** essa estrutura em uma tabela plana
facilitando o consumo por modelos de machine learning e ferramentas de análise.

O arquivo final `features.parquet` contém todas as features extraídas nos dois estágios,
consolidadas em um único schema tabular com colunas como `label`, `conf`, `bbox`, `r_mean`,
`s_mean`, `v_mean`, `ocr_text`, `ocr_global_text` e `embedding`.

Esta etapa é equivalente à fase de *dataset preprocessing* descrita na Seção 3.2 do artigo, onde as features brutas são organizadas para análise estatística e modelagem.

In [ ]:
from feature_postprocess import run as run_postprocess

print('Executando pos-processamento...')
run_postprocess(
    input_path=STAGE_B_PARQUET,
    output_path=FINAL_PARQUET,
)
print(f'\nPos-processamento concluido. Saida: {FINAL_PARQUET}')

In [ ]:
# Validacao final
if os.path.exists(FINAL_PARQUET):
    df_final = pd.read_parquet(FINAL_PARQUET)
    print(f'Arquivo final: {FINAL_PARQUET}')
    print(f'  Linhas (deteccoes): {len(df_final)}')
    print(f'  Colunas: {df_final.columns.tolist()}')
    display(df_final.head(5))
elif os.path.exists(STAGE_B_PARQUET):
    print('Arquivo final nao encontrado, exibindo dados do Stage B:')
    if 'df_b' in locals() and len(df_b) > 0:
        display(df_b.head(5))
else:
    print('Nenhum dado processado disponivel. Execute os estagios anteriores primeiro.')

---
## Sumário Final

Visão consolidada do pipeline completo: volumes processados, cobertura de OCR,
distribuição de labels e amostras visuais.

In [ ]:
print('=' * 60)
print('RESUMO DO PIPELINE DE FEATURES')
print('=' * 60)

total_images = len(all_images) if all_images else 0
print(f'\nImagens fonte: {total_images}')
print(f'Subreddits: {len(subreddits)}')

if os.path.exists(STAGE_A_PARQUET):
    print(f'\n[Estagio A]')
    a_images = len(pd.read_parquet(STAGE_A_PARQUET))
    print(f'  Imagens processadas: {a_images}')
    if 'df_a' in locals() and len(df_a) > 0:
        print(f'  Total deteccoes: {len(df_a)}')
        print(f'  Labels unicas: {df_a["label"].nunique() if "label" in df_a.columns else "N/A"}')
        if 'status' in df_a.columns:
            ok = (df_a['status'] == 'ok').sum()
            print(f'  Deteccoes OK: {ok}/{len(df_a)} ({100*ok/len(df_a):.1f}%)')

if os.path.exists(STAGE_B_PARQUET):
    print(f'\n[Estagio B]')
    if 'df_b' in locals() and len(df_b) > 0:
        print(f'  Total deteccoes enriquecidas: {len(df_b)}')
        if 'ocr_text' in df_b.columns:
            pct = (df_b['ocr_text'].fillna('').str.strip() != '').mean() * 100
            print(f'  OCR crop-level coverage: {pct:.1f}%')
        if 'ocr_global_text' in df_b.columns:
            pct_g = (df_b['ocr_global_text'].fillna('').str.strip() != '').mean() * 100
            print(f'  OCR global coverage: {pct_g:.1f}%')

print(f'\n[Pos-processamento]')
if os.path.exists(FINAL_PARQUET):
    n = len(pd.read_parquet(FINAL_PARQUET))
    print(f'  Tabela normalizada: {n} linhas')
    print(f'  Path: {FINAL_PARQUET}')
elif os.path.exists(STAGE_B_PARQUET):
    print(f'  Pos-processamento pendente (execute a celula de pos-processamento)')
else:
    print(f'  Pipeline nao executado')

print('\n' + '=' * 60)
print('Pipeline concluido com sucesso!')